In [ ]:
# pip install transformers accelerate peft trl bitsandbytes datasets evaluate

In [ ]:
import torch

from google.colab import userdata

from datasets import load_dataset
from huggingface_hub import login
from peft import (
    AutoPeftModelForCausalLM,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from trl import SFTConfig, SFTTrainer, setup_chat_format

In [ ]:
login(userdata.get("HF_TOKEN"))

In [ ]:
dataset = load_dataset(path="FreedomIntelligence/medical-o1-reasoning-SFT", name="en", split="train[:500]")

dataset

Dataset({
    features: ['Question', 'Complex_CoT', 'Response'],
    num_rows: 500
})

In [ ]:
dataset[0]

{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the l

In [ ]:
def row_to_messages(q, cot, resp):
  assistant_content = f"<think>\n{cot}\n</think>\n{resp}"
  return [
      {"role": "user", "content": q},
      {"role": "assistant", "content": assistant_content},
  ]

dataset = dataset.map(
    lambda ex: {"messages": row_to_messages(ex["Question"], ex["Complex_CoT"], ex["Response"])},
    remove_columns=[c for c in dataset.column_names if c not in []]
)

In [ ]:
dataset[0]

{'messages': [{'content': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
   'role': 'user'},
  {'content': "<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood cl

In [ ]:
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
).to(device)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name, trust_remote_code=True, use_fast=True)
tokenizer.padding_side = "right"

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
tokenizer.chat_template

"{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}{%- for message in messages %}{%- if message['role'] == 'system' %}{% set ns.system_prompt = message['content'] %}{%- endif %}{%- endfor %}{{bos_token}}{{ns.system_prompt}}{%- for message in messages %}{%- if message['role'] == 'user' %}{%- set ns.is_tool = false -%}{{'<｜User｜>' + message['content']}}{%- endif %}{%- if message['role'] == 'assistant' and message['content'] is none %}{%- set ns.is_tool = false -%}{%- for tool in message['tool_calls']%}{%- if not ns.is_first %}{{'<｜Assistant｜><｜tool▁calls▁begin｜><｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\\n' + '```json' + '\\n' + tool['function']['arguments'] + '\\n' + '```' + '<｜tool▁call▁end｜>'}}{%- set ns.is_first = true -%}{%- else %}{{'\\n' + '<｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['fu

In [ ]:
tokenizer.chat_template = """{# --- Defaults & scratch space --- #}
{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}
{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}

{# --- Collect system prompt (if present) --- #}
{%- for message in messages -%}
  {%- if message['role'] == 'system' -%}
    {% set ns.system_prompt = message['content'] %}
  {%- endif -%}
{%- endfor -%}

{# --- BOS + system prompt --- #}
{{ bos_token }}{{ ns.system_prompt }}

{# --- Render conversation --- #}
{%- for message in messages -%}

  {# --- User --- #}
  {%- if message['role'] == 'user' -%}
    {%- set ns.is_tool = false -%}
    {{ '<｜User｜>' + message['content'] }}
  {%- endif -%}

  {# --- Assistant (tool-calls) --- #}
  {%- if message['role'] == 'assistant' and message['content'] is none -%}
    {%- set ns.is_tool = false -%}
    {%- for tool in message['tool_calls'] -%}
      {%- if not ns.is_first -%}
        {{ '<｜Assistant｜><｜tool▁calls▁begin｜><｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\n' + '```json' + '\n' + tool['function']['arguments'] + '\n' + '```' + '<｜tool▁call▁end｜>' }}
        {%- set ns.is_first = true -%}
      {%- else -%}
        {{ '\n' + '<｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\n' + '```json' + '\n' + tool['function']['arguments'] + '\n' + '```' + '<｜tool▁call▁end｜>' }}
        {{ '<｜tool▁calls▁end｜><｜end▁of▁sentence｜>' }}
      {%- endif -%}
    {%- endfor -%}
  {%- endif -%}

  {# --- Assistant (normal text: CoT + final answer) --- #}
  {%- if message['role'] == 'assistant' and message['content'] is not none -%}
    {%- if ns.is_tool -%}
      {{ '<｜tool▁outputs▁end｜>' }}
      {{ '<｜Assistant｜>' }}
      {% generation %}
      {{ message['content'] }}
      {% endgeneration %}
      {{ '<｜end▁of▁sentence｜>' }}
      {%- set ns.is_tool = false -%}
    {%- else -%}
      {{ '<｜Assistant｜>' }}
      {% generation %}
      {{ message['content'] }}
      {% endgeneration %}
      {{ '<｜end▁of▁sentence｜>' }}
    {%- endif -%}
  {%- endif -%}

  {# --- Tool outputs --- #}
  {%- if message['role'] == 'tool' -%}
    {%- set ns.is_tool = true -%}
    {%- if ns.is_output_first -%}
      {{ '<｜tool▁outputs▁begin｜><｜tool▁output▁begin｜>' + message['content'] + '<｜tool▁output▁end｜>' }}
      {%- set ns.is_output_first = false -%}
    {%- else -%}
      {{ '\n<｜tool▁output▁begin｜>' + message['content'] + '<｜tool▁output▁end｜>' }}
    {%- endif -%}
  {%- endif -%}

{%- endfor -%}

{# --- Close any open tool outputs --- #}
{% if ns.is_tool %}{{ '<｜tool▁outputs▁end｜>' }}{% endif %}

{# --- Optional generation prompt --- #}
{% if add_generation_prompt and not ns.is_tool -%}
  {{ '<｜Assistant｜>' }}
  {% generation %}{% endgeneration %}
{%- endif %}"""

In [ ]:
tokenizer.chat_template

"{# --- Defaults & scratch space --- #}\n{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}\n{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}\n\n{# --- Collect system prompt (if present) --- #}\n{%- for message in messages -%}\n  {%- if message['role'] == 'system' -%}\n    {% set ns.system_prompt = message['content'] %}\n  {%- endif -%}\n{%- endfor -%}\n\n{# --- BOS + system prompt --- #}\n{{ bos_token }}{{ ns.system_prompt }}\n\n{# --- Render conversation --- #}\n{%- for message in messages -%}\n\n  {# --- User --- #}\n  {%- if message['role'] == 'user' -%}\n    {%- set ns.is_tool = false -%}\n    {{ '<｜User｜>' + message['content'] }}\n  {%- endif -%}\n\n  {# --- Assistant (tool-calls) --- #}\n  {%- if message['role'] == 'assistant' and message['content'] is none -%}\n    {%- set ns.is_tool = false -%}\n    {%- for tool in message['tool_calls'] -%}\n      {%- if not ns.is_first -%}\n        {{ 

In [ ]:
finetune_name = "DeepSeek-R1-Distill-Qwen-1.5B0-medical-o1-reasoning-SFT-QLoRA"

In [ ]:
rank_dimension = 64
lora_alpha = 16
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 73,859,072 || all params: 1,850,947,072 || trainable%: 3.9903


In [ ]:
max_length = 4096

args = SFTConfig(
  output_dir=finetune_name,
  num_train_epochs=1,
  per_device_train_batch_size=2,
  gradient_accumulation_steps=2,
  gradient_checkpointing=True,
  optim="paged_adamw_8bit",
  learning_rate=2e-4,
  max_grad_norm=0.3,
  warmup_ratio=0.03,
  lr_scheduler_type="cosine",
  logging_steps=10,
  save_steps=500,
  save_total_limit=2,
  save_strategy="epoch",
  bf16=True,
  push_to_hub=False,
  report_to="none",
  max_length = max_length,
  packing=False,
  dataset_kwargs={
      "add_special_tokens": False,
      "append_concat_token": False,
  },
  assistant_only_loss=True,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

trainer.save_model()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.382700
20,2.315900
30,2.392100
40,2.303900
50,2.291600
60,2.386200
70,2.280200
80,2.300300
90,2.226000
100,2.273900


In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    args.output_dir, safe_serialization=True, max_shard_size="2GB"
)

In [ ]:
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=finetune_name)
model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name, device_map="auto", dtype=torch.bfloat16
)
pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer, device=device)

Device set to use cuda


In [ ]:
prompts = [
    "A 21-year-old man presents with painful urination and watery discharge from the penis, which is progressively worsening. His physical examination reveals a tender urethra with discharge, and a gram stain of the discharge is negative for bacteria but shows many neutrophils. What is the most likely infectious cause of his symptoms?",
    "In a 7-month-old child diagnosed with H. influenzae meningitis, what investigation should be conducted during follow-up to assess potential complications related to hearing?",
    "What is the treatment of choice for a 40-year-old primiparous woman diagnosed with endometriosis during diagnostic laparoscopy, where findings include normal uterus, chocolate cysts on both ovaries, endometriotic deposits on the round ligament on the right side, both fallopian tubes, and the pouch of Douglas, as well as moderately dense adhesions between the fallopian tubes and the pouch of Douglas?",
    "What is the most likely underlying mechanism causing thrombocytopenia in a 25-year-old woman with systemic lupus erythematosus, presenting with diffuse petechiae, fatigue, lymphadenopathy, splenomegaly, normocytic anemia, and low platelet count?",
    """A patient presents with hyperacusis, loss of lacrimation and loss of taste sensation in the anterior 2/3rd of the tongue. Oedema extends up to which level of facial nerve -
A. Vertical part
B. Vertical part proximal to nerve to stapedius
C. Vertical part and beyond nerve to stapedius
D. Proximal to geniculate ganglion""",
]

GEN_KW = dict(
    max_new_tokens=1024,
    min_new_tokens=32,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.10,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.convert_tokens_to_ids("<｜end▁of▁sentence｜>"),
)

def test_inference(prompt):
  prompt = pipe.tokenizer.apply_chat_template(
      [{"role": "user", "content": prompt}],
      tokenize=False,
      add_generation_prompt=True
  )
  outputs = pipe(prompt, **GEN_KW)
  return outputs[0]["generated_text"][len(prompt):].strip()


for prompt in prompts:
  print(f"  prompt:\n{prompt}")
  print(f"  response:\n{test_inference(prompt)}")
  print("-" * 50)

  prompt:
A 21-year-old man presents with painful urination and watery discharge from the penis, which is progressively worsening. His physical examination reveals a tender urethra with discharge, and a gram stain of the discharge is negative for bacteria but shows many neutrophils. What is the most likely infectious cause of his symptoms?
  response:
Okay, so I'm trying to figure out what's going on here. A 20-something guy has painful urinary tract infections, specifically with discharge that's both watery and from the penile area. The physical exam shows a tender, discharge-filled urethron, and when they did a gram test, there were no bacteria, but lots of neutrophiles. Hmm.

First off, I remember that the urinary tract infection (UTI) can be caused by various things like bacteria, viruses, or parasites. Since the discharge isn't bacterial, it's probably not something like E. coli or Salmonella. But why are there neutrophil-like findings? Neutrophils are usually part of the immune r